# Step 1: Setup prerequisites

- Set the LLM provider and passkey provided by your workshop instructor

- `LLM_PROVIDER` can be set to one of "aws"/ "microsoft" / "google"

In [ ]:
# NOTE: LLM_PROVIDER can be set to one of "aws"/ "microsoft" / "google"
LLM_PROVIDER = "aws"
PASSKEY = "replace-with-passkey"

In [ ]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

# ----- MONGODB SETUP -----
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-ai-agents")
# Check the connection to the server
mongodb_client.admin.command("ping")

# ----- API KEY SETUP -----
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env([LLM_PROVIDER, "voyageai"], PASSKEY)

# Step 2: Import data into MongoDB

In [ ]:
import json

### **Do not change the values assigned to the variables below**

In [ ]:
#  Database name
DB_NAME = "mongodb_genai_devday_agents"
# Name of the collection to store flights data
FLIGHTS_COLLECTION_NAME = "flights"
# Name of the collection to store AirBnB listings data
LISTINGS_COLLECTION_NAME = "listings"
# Name of the collection to store bookings data
BOOKINGS_COLLECTION_NAME = "bookings"


In [ ]:
# Connect to the `LISTINGS_COLLECTION_NAME` collection
listings_collection = mongodb_client[DB_NAME][LISTINGS_COLLECTION_NAME]
# Connect to the `FLIGHTS_COLLECTION_NAME` collection
flights_collection = mongodb_client[DB_NAME][FLIGHTS_COLLECTION_NAME]
# Connect to the `BOOKINGS_COLLECTION_NAME` collection
bookings_collection = mongodb_client[DB_NAME][BOOKINGS_COLLECTION_NAME]


In [ ]:
# Insert a dataset of AirBnB listings into the `LISTINGS_COLLECTION_NAME` collection
with open(f"../data/{LISTINGS_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {LISTINGS_COLLECTION_NAME} collection.")
listings_collection.delete_many({})
listings_collection.insert_many(data)
print(
    f"{listings_collection.count_documents({})} documents ingested into the {LISTINGS_COLLECTION_NAME} collection."
)

In [ ]:
# Insert a dataset of flight routes into the `FLIGHTS_COLLECTION_NAME` collection
with open(f"../data/{FLIGHTS_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {FLIGHTS_COLLECTION_NAME} collection.")
flights_collection.delete_many({})
flights_collection.insert_many(data)
print(
    f"{flights_collection.count_documents({})} documents ingested into the {FLIGHTS_COLLECTION_NAME} collection."
)

# Step 3: Instantiate the LLM

In [ ]:
from utils import get_llm

In [ ]:
# Obtain the Langchain LLM object using the `get_llm` function from the `utils` module
llm = get_llm(LLM_PROVIDER)

# Step 4: Create agent tools


In [ ]:
from langchain_mongodb.agent_toolkit.database import MongoDBDatabase
from langchain_mongodb.agent_toolkit.toolkit import MongoDBDatabaseToolkit

### Create a custom tool

In [ ]:
from datetime import datetime, timezone
from typing import Literal
from langchain.tools import tool

In [ ]:
# Define a custom tool to save booking details to MongoDB
@tool
def create_booking(
    booking_type: Literal["flight", "accommodation"],
    traveler_name: str,
    destination: str,
    item_name: str,
    price_usd: float,
    travelers: int = 1,
) -> str:
    """Save booking details to the bookings collection.

    Args:
        booking_type: Either "flight" or "accommodation".
        traveler_name: Full name of the traveler making the booking.
        destination: Destination city, e.g. "Barcelona".
        item_name: Flight number for flights, or listing name for accommodation.
        price_usd: Total price of this booking in USD.
        travelers: Number of travelers. Defaults to 1.
    """
    # Build the booking document to insert into MongoDB
    booking = {
        "booking_type": booking_type,
        "traveler_name": traveler_name,
        "destination": destination,
        "item_name": item_name,
        "price_usd": price_usd,
        "travelers": travelers,
        "created_at": datetime.now(timezone.utc),
    }
    # Insert the booking document into the `bookings` collection
    bookings_collection.insert_one(booking)
    return f"Booked {booking_type} to {destination}: {item_name}. Total price ${price_usd:.2f} for {travelers} travelers."

### Import tools from the MongoDB Database Toolkit

In [ ]:
# Access the `DB_NAME` database using the `MongoDBDatabase` class
db = MongoDBDatabase.from_connection_string(connection_string=MONGODB_URI, database=DB_NAME)

In [ ]:
# Initialize the MongoDB database toolkit with the `db` and `llm` defined previously
toolkit = MongoDBDatabaseToolkit(db=db, llm=llm)

### Investigate the tools

In [ ]:
# Get the list of tools from the MongoDB database toolkit
tools = toolkit.get_tools()
# Append the `create_booking` tool to the list of tools obtained from the MongoDB database toolkit
tools.append(create_booking)

In [ ]:
# Investigate the tool names, descriptions and arguments for each tool
tools_map = {t.name: t for t in tools}

for name, t in tools_map.items():
    print(f"{name}\n  description: {t.description}\n  args: {t.args}\n")

In [ ]:
# Test out the `mongodb_list_collections` tool
# The below tool call should return the list of collections in the `DB_NAME` database
tools_map["mongodb_list_collections"].invoke("")

In [ ]:
# Test out the `mongodb_schema` tool
# The below tool call should return the schema of documents in the `listings` collection and 3 sample documents from it
print(tools_map["mongodb_schema"].invoke("listings"))

In [ ]:
# Test the `mongodb_query_checker` tool
# The below tool call takes a MongoDB `query` as input, corrects it if necessary and returns the corrected query
query = 'db.flights.aggregate([{"$match": {"to_city": "Barcelona",}}, {"$limit": 3}])'
tools_map["mongodb_query_checker"].invoke(query)

In [ ]:
# Test the `mongodb_query` tool
# The below tool call executes a MongoDB `query` against the `flights` collection and returns the results
query = [
    {"$match": {"to_city": "Barcelona"}},
    {"$limit": 3}
]
print(tools_map["mongodb_query"].invoke(f"db.flights.aggregate({json.dumps(query)})"))

In [ ]:
# Test the `create_booking` tool
# The below tool call writes a booking documents to the `bookings` collection
create_booking.invoke({
    "booking_type": "flight",
    "traveler_name": "Sam Rivera",
    "destination": "Barcelona",
    "item_name": "TA303",
    "price_usd": 230.96,
    "travelers": 2
})

# Step 5: Create the LLM prompt

In addition to the tools, the MongoDB database toolkit also provides an LLM system prompt containing guidance on how to use the tools available in the toolkit. However, we will create our own system prompt since we want to provide additional tools and instructions to the agent.

You can access the system prompt available in the toolkit as follows:
```
from langchain_mongodb.agent_toolkit import MONGODB_AGENT_SYSTEM_PROMPT
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
# Create a system prompt for the agent providing instructions on how to use the database tools
SYSTEM_PROMPT = """
You are an agent designed to interact with a MongoDB database. 
You have access to the following tools: {tool_names}.
Only use the information returned by the tools to construct your final answer.

Most tools are read-only, but you can perform write actions to create new bookings and store them to MongoDB using the `create_booking` tool.

ALWAYS start by looking at the collections in the database to see what you can query, unless you have this information in memory.
Then you should query the schema of the most relevant collections.
To get data from MongoDB, create a syntactically correct MongoDB query to run, analyze the results of the query and decide what to do next.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant field to return the most interesting examples in the database.
Never query for all the fields from a specific collection, only query for the relevant fields given the question.
Read queries MUST include the collection name and the contents of the aggregation pipeline. An example query looks like:

```python
db.flights.aggregate([{{"\$match": {{"to_city": "Barcelona"}}}}, {{"\$limit": 3}}])
```

Do not re-run tools unless absolutely necessary. If you are not able to get enough information using the tools, reply with I DON'T KNOW.
"""

In [ ]:
# Create a prompt template which includes the system prompt and a placeholder for the `messages` i.e. user queries and agent responses
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [ ]:
# Pre-fill `top_k` and `tool_names` in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))

# Step 6: Give the LLM access to tools

📚 https://docs.langchain.com/oss/python/langgraph/quickstart#1-define-tools-and-model

In [ ]:
# Bind the `tools` defined in Step 4 to the `llm` instantiated in Step 3
bind_tools = llm.bind_tools(tools)

📚 https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/pipe (See Example)

In [ ]:
# Chain the `prompt` with the tool-augmented_llm using the `|` operator
llm_with_tools = prompt | bind_tools

In [ ]:
# Test that the LLM is making the right tool calls
llm_with_tools.invoke(
    ["I want to go to Barcelona. What's the cheapest nonstop flight there?"]
).tool_calls

The above test shows that the LLM will first call the `mongodb_list_collections` tool, given any query.

This is exactly what we want the agent to do as the first step so it can better understand our data.

# Step 7: Define graph state

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

In [ ]:
# Define the graph state
# We are only tracking chat messages but you can track also other attributes
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]

# Step 7: Define graph nodes

In [ ]:
from langchain_core.messages import ToolMessage
from typing import Dict, List

In [ ]:
# Define the agent node
def agent_node(state: GraphState) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to messages
    """
    # Get the messages from the graph `state`
    messages = state["messages"]
    # Invoke `llm_with_tools` with `messages` using the `invoke` method
    # HINT: See Step 6 for how to invoke `llm_with_tools`
    result = llm_with_tools.invoke(messages)
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Define tool node
def tool_node(state: GraphState) -> Dict[str, List]:
    """
    Tool node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to messages
    """
    result = []
    # Get the list of tool calls from messages
    tool_calls = state["messages"][-1].tool_calls
    # A tool_call looks as follows:
    # {
    #     "name": "get_information_for_question_answering",
    #     "args": {"user_query": "What are Atlas Triggers"},
    #     "id": "call_H5TttXb423JfoulF1qVfPN3m",
    #     "type": "tool_call",
    # }
    # Iterate through `tool_calls`
    for tool_call in tool_calls:
        # Get the tool from `tools_map` defined in Step 4, using the `name` attribute of the `tool_call`
        tool = tools_map[tool_call["name"]]
        # Invoke the `tool` using the `args` attribute of the `tool_call`
        # HINT: See previous line to see how to extract attributes from the `tool_call`
        observation = tool.invoke(tool_call["args"])
        # Append the result of executing the tool to the `result` list as a ToolMessage
        # The `content` of the message is `observation` i.e. result of the tool call
        # The `tool_call_id` can be obtained from the `tool_call`
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": result}

# Step 8: Define conditional edges

In [ ]:
from langgraph.graph import END

In [ ]:
# Define conditional routing function
def route_tools(state: GraphState):
    """
    Use in the conditional_edge to route to the tool node if the last message
    has tool calls. Otherwise, route to the end.
    """
    # Get messages from graph state
    messages = state.get("messages", [])
    if len(messages) > 0:
        # Get the last AI message from messages
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    # Check if the last message has tool calls
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        # If yes, return "tools"
        return "tools"
    # If no, return END
    return END

# Step 9: Build the graph

In [ ]:
from langgraph.graph import StateGraph, START

In [ ]:
# Instantiate the graph
graph = StateGraph(GraphState)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#nodes

In [ ]:
# Add nodes to the `graph` using the `add_node` function
# Add a `agent` node. The `agent` node should run the `agent_node` function
graph.add_node("agent", agent_node)
# Add a `tools` node. The `tools` node should run the `tool_node` function
graph.add_node("tools", tool_node)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#normal-edges

In [ ]:
# Add fixed edges to the `graph` using the `add_edge` method
# Add an edge from the START node to the `agent` node
graph.add_edge(START, "agent")
# Add an edge from the `tools` node to the `agent` node
graph.add_edge("tools", "agent")

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges

In [ ]:
# Use the `add_conditional_edges` method to add a conditional edge from the `agent` node to the `tools` node
# based on the output of the `route_tools` function
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# Compile the `graph`
app = graph.compile()

In [ ]:
# Visualize the graph
app

# Step 10: Execute the graph

In [ ]:
# Define a function to execute the graph and stream outputs from each step
def execute_graph(user_input: str) -> None:
    """
    Stream outputs from the graph execution

    Args:
        user_input (str): User query string
    """
    # Stream outputs from each step in the graph
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # Stream full value of the state after each step
        stream_mode="values",
    ):
        # Print the latest message from the step
        step["messages"][-1].pretty_print()

In [ ]:
# Test the graph execution to view end-to-end flow
execute_graph("I want to go to Barcelona. What's the cheapest nonstop flight there, and the cheapest place to stay for 2 people?")

In [ ]:
# Ask a follow up question
# Notice that the agent does not remember the previous coonversation since we have no way to persist it yet
execute_graph("What did I just ask you?")

# Step 11: Add short-term memory to the agent

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

In [ ]:
# Initialize a MongoDB checkpointer
checkpointer = MongoDBSaver(mongodb_client)

In [ ]:
# Instantiate the graph with the checkpointer
app = graph.compile(checkpointer=checkpointer)

📚 https://docs.langchain.com/oss/python/langgraph/persistence#threads

In [ ]:
# Update the graph execution function to handle thread IDs
def execute_graph(thread_id: str, user_input: str) -> None:
    """
    Stream outputs from the graph execution

    Args:
        thread_id (str): Thread ID for the checkpointer
        user_input (str): User query string
    """
    # Create a runtime config containing the thread ID
    config = {"configurable": {"thread_id": thread_id}}
    # Stream outputs from each step in the graph
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # Pass the config as an additional parameter to the graph invocation
        config,
        stream_mode="values",
    ):
        # Print the latest message from the step
        step["messages"][-1].pretty_print()

In [ ]:
# Test graph execution with a thread ID
execute_graph(
    "1",
    "I want to go to Barcelona. What's the cheapest nonstop flight there, and the cheapest place to stay for 2 people?",
)

In [ ]:
# Ask a follow-up question to ensure short-term memory works
# Notice it uses the flight and accommodation information from the previous question
execute_graph(
    "1",
    "Book the flight and accommodation for 3 nights under the name Jane Doe.",
)

# 🦹‍♀️ Step 12: Add long-term memory to the agent

📚 https://docs.langchain.com/oss/python/langgraph/add-memory#use-semantic-search

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.store.mongodb import MongoDBStore, create_vector_index_config
from langchain_voyageai import VoyageAIEmbeddings
import uuid

In [ ]:
# Initialize a MongoDB collection for long-term memory storage
memory_collection = mongodb_client[DB_NAME]["memories"]

In [ ]:
# Initialize a MongoDB long-term memory store with Voyage embeddings to retrieve memories using vector search
mongodb_store = MongoDBStore(
    collection=memory_collection,
    index_config=create_vector_index_config(
        embed=VoyageAIEmbeddings(model="voyage-4"),
        dims=1024,
    ),
)

In [ ]:
# Create a tool to save memories to the long-term memory store
@tool
def save_memory(memory: str, config: RunnableConfig) -> str:
    """
    Save important facts and preferences about the user for future conversations.

    Args:
    memory: The information to remember
    """
    user_id = config["configurable"]["user_id"]
    mongodb_store.put(
        # Namespace for the memory entry. You can also have sub-namespaces to store different types of memories, eg: ("user_1", "preferences")
        # Has to be a tuple, even if it contains empty values
        (user_id,),
        # Unique memory ID
        key=str(uuid.uuid4()),
        # Content of the memory-- needs to be a dictionary
        value={"text": memory},
    )
    return f"Memory saved: {memory}"

In [ ]:
# Create a memory prompt providing instructions on what memories to extract and how to use the `save_memory` tool
MEMORY_PROMPT = """
Whenever the user states a preference, constraint, or fact about themselves, such as budget, party size, accommodation type etc, call `save_memory` to record it to long-term memory.
Save one fact/preference per memory entry. Do not save the same fact/preference multiple times.
Use past user preferences to personalize future conversations.
Past user memories:\n{memories}
"""

In [ ]:
# Update the tools list to include the `save_memory` tool
tools.append(save_memory)
tools_map = {t.name: t for t in tools}
# Bind the updated tool list to the LLM
bind_tools = llm.bind_tools(tools)
# Update the system prompt to include the `MEMORY_PROMPT` and create a new prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT + MEMORY_PROMPT),
        MessagesPlaceholder(variable_name="messages")
    ]
)
# Pre-fill `top_k` and `tool_names` in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))
# Chain the `prompt` with the tool-augmented LLM
llm_with_tools = prompt | bind_tools

In [ ]:
# Update the agent node to retrieve relevant long-term memories when responding
def agent_node(state: GraphState, config: RunnableConfig) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state
        config (RunnableConfig): Runtime config containing the `user_id` for memory retrieval

    Returns:
        Dict[str, List]: Updates to messages
    """
    # Get `messages` from the graph `state`
    messages = state["messages"]
    # Get the `user_id` from the runtime `config`
    user_id = config["configurable"]["user_id"]
    # Search for relevant long-term memories using the user's last message
    memories = mongodb_store.search((user_id,), query=messages[-1].content, limit=10)
    # Format retrieved memories into a string.
    memories = "\n".join(f"- {m.value['text']}" for m in memories) or "No memories yet."
    # Invoke the tool-augmented LLM with the `memories` and `messages` (including chat history) to generate a response
    result = llm_with_tools.invoke(
        {
            "memories": memories,
            "messages": messages,
        }
    )
    # Write the `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Update the tool node to pass the runtime config to the tools
def tool_node(state: GraphState, config: RunnableConfig) -> Dict[str, List]:
    """
    Tool node

    Args:
        state (GraphState): Graph state
        config (RunnableConfig): Runtime config
    Returns:
        Dict[str, List]: Updates to messages
    """
    result = []
    # Get the list of tool calls from messages
    tool_calls = state["messages"][-1].tool_calls
    # Iterate through `tool_calls`
    for tool_call in tool_calls:
        # Get the tool from `tools_map` defined in Step 4, using the `name` attribute of the `tool_call`
        tool = tools_map[tool_call["name"]]
        # Invoke the `tool` using the `args` attribute of the `tool_call` and the runtime `config`
        observation = tool.invoke(tool_call["args"], config)
        # Append the result of executing the tool to the `result` list as a ToolMessage
        # The `content` of the message is `observation` i.e. result of the tool call
        # The `tool_call_id` can be obtained from the `tool_call`
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": result}

In [ ]:
# Rebuild the agent graph
graph = StateGraph(GraphState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# Compile the graph with the MongoDB checkpointer for short-term memory as well as the MongoDB memory store for long-term memory
app = graph.compile(checkpointer=checkpointer, store=mongodb_store)

In [ ]:
# Update the graph execution function to handle user IDs to organize long-term memory
def execute_graph(thread_id: str, user_id: str, user_input: str) -> None:
    """
    Stream outputs from the graph execution

    Args:
        thread_id (str): Thread ID for the checkpointer
        user_id (str): User ID to organize long-term memory
        user_input (str): User query string
    """
    # Create a runtime config containing the thread ID and user ID
    # The `thread_id` will need to be scoped by `user_id` to avoid memory leakage between users
    config = {"configurable": {"thread_id": f"{thread_id}-{user_id}", "user_id": user_id}}
    # Stream outputs from each step in the graph
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # Pass the config as an additional parameter
        config,
        stream_mode="values",
    ):
        # Print the latest message from the step
        step["messages"][-1].pretty_print()

In [ ]:
# Test creating memories in thread `1` for user `Jane`
# Notice the `save_memory` tool being invoked to save the user's preferences

execute_graph(
    "1",
    "Jane",
    "Remember that I only want entire homes and never spend more than $100 a night on stays."
)

In [ ]:
# Test cross-session long-term memory persistence by starting a new thread `2` for user `Jane`
# Notice that the agent is able to recall the user's preferences from the previous thread and use them to answer the question
execute_graph(
    "1",
    "Jane",
    "I want to go to Barcelona. Can you find me places to stay?"
)

In [ ]:
# Test that memories aren't leaking across users-- Jane's preferences should not be used for user `John`
# Notice that the agent doesn't use information from Jane's previous conversation to answer John's question
execute_graph(
    "5",
    "John",
    "I want to go to Barcelona. Can you find me places to stay?"
)